# 💊 MedCheck AI - Complete Training Pipeline

**What this notebook does:**
1. 📥 Download FDA FAERS real adverse event data
2. 🔧 Process into drug interaction training dataset
3. 🤖 Train XGBoost ML model (GPU accelerated)
4. 📚 Build RAG Vector Database (FAISS)
5. 🧠 Integrate Groq LLM with RAG validation
6. 💾 Save everything to Google Drive
7. ✅ Test complete pipeline

**Data Sources:**
- FDA FAERS: 10M+ real adverse event reports (US Government)
- PubMed: 30M+ peer-reviewed clinical papers (NIH)

> ⚡ Enable GPU: Runtime → Change runtime type → T4 GPU

## ⚙️ STEP 0: Setup & Install

In [ ]:
# Install all required packages
!pip install -q faiss-gpu sentence-transformers xgboost scikit-learn \
    pandas numpy requests groq tqdm joblib

print('✅ All packages installed!')

In [ ]:
# Mount Google Drive (to save models)
from google.colab import drive
drive.mount('/content/drive')

import os
# Create project folder in Drive
PROJECT_DIR = '/content/drive/MyDrive/MedCheck_AI'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/rag', exist_ok=True)

print(f'✅ Project folder created: {PROJECT_DIR}')

In [ ]:
# Check GPU
import torch
print(f'GPU Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 📥 STEP 1: Download FDA FAERS Data

In [ ]:
import requests
import json
import zipfile
import os
from tqdm import tqdm

def download_fda_file(url, output_path):
    """Download FDA file with progress bar"""
    response = requests.get(url, stream=True)
    total = int(response.headers.get('content-length', 0))

    with open(output_path, 'wb') as f, tqdm(
        desc=os.path.basename(output_path),
        total=total,
        unit='iB',
        unit_scale=True
    ) as bar:
        for chunk in response.iter_content(chunk_size=8192):
            size = f.write(chunk)
            bar.update(size)

# Download FDA FAERS data (first 3 files = ~600K records)
FDA_FILES = [
    'https://download.open.fda.gov/drug/event/drug-event-0001-of-0014.json.zip',
    'https://download.open.fda.gov/drug/event/drug-event-0002-of-0014.json.zip',
    'https://download.open.fda.gov/drug/event/drug-event-0003-of-0014.json.zip',
]

DATA_DIR = '/content/fda_data'
os.makedirs(DATA_DIR, exist_ok=True)

for i, url in enumerate(FDA_FILES, 1):
    filename = url.split('/')[-1]
    output_path = f'{DATA_DIR}/{filename}'

    if os.path.exists(output_path.replace('.zip', '')):
        print(f'✅ File {i} already exists, skipping...')
        continue

    print(f'\n📥 Downloading file {i}/3...')
    download_fda_file(url, output_path)

    print(f'📦 Unzipping...')
    with zipfile.ZipFile(output_path, 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)
    os.remove(output_path)
    print(f'✅ File {i} ready!')

# List files
files = [f for f in os.listdir(DATA_DIR) if f.endswith('.json')]
print(f'\n✅ Total files downloaded: {len(files)}')
for f in files:
    size = os.path.getsize(f'{DATA_DIR}/{f}') / 1e6
    print(f'  {f}: {size:.1f} MB')

## 🔧 STEP 2: Process FDA Data into Training Dataset

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
import re

def clean_drug_name(name):
    """Clean drug name - remove dosage, special chars"""
    # Remove dosage: 10mg, 500MG, 81 mg etc.
    name = re.sub(r'\s*\d+\.?\d*\s*(mg|mcg|ml|g|iu|units?)\b', '', name, flags=re.IGNORECASE)
    # Remove numbers at end
    name = re.sub(r'\s+\d+\s*$', '', name)
    # Remove special characters
    name = re.sub(r'[^a-zA-Z\s-]', '', name)
    # Clean whitespace
    name = ' '.join(name.split()).strip().lower()
    return name

def process_fda_file(filepath):
    """Process one FDA FAERS file"""
    print(f'Processing {os.path.basename(filepath)}...')

    with open(filepath, 'r') as f:
        data = json.load(f)

    results = data.get('results', [])
    print(f'  Found {len(results):,} adverse event reports')

    drug_pairs = defaultdict(lambda: {
        'total_reports': 0,
        'serious_reports': 0,
        'death_reports': 0,
        'hospitalization_reports': 0,
        'reactions': defaultdict(int)
    })

    for report in tqdm(results, desc='Processing reports'):
        patient = report.get('patient', {})
        drugs = patient.get('drug', [])
        reactions = patient.get('reaction', [])

        # Need at least 2 drugs
        if len(drugs) < 2:
            continue

        # Extract + clean drug names
        drug_names = []
        for drug in drugs:
            name = drug.get('medicinalproduct', '').strip()
            cleaned = clean_drug_name(name)
            if cleaned and len(cleaned) > 2 and cleaned.isalpha() or '-' in cleaned:
                drug_names.append(cleaned)

        drug_names = list(set(drug_names))  # Remove duplicates

        if len(drug_names) < 2:
            continue

        # Seriousness flags
        is_serious = int(report.get('serious', 0) or 0)
        is_death = int(report.get('seriousnessdeath', 0) or 0)
        is_hosp = int(report.get('seriousnesshospitalization', 0) or 0)

        # Create all pairs
        for i in range(len(drug_names)):
            for j in range(i + 1, len(drug_names)):
                d1, d2 = sorted([drug_names[i], drug_names[j]])
                key = f'{d1}|||{d2}'

                drug_pairs[key]['total_reports'] += 1
                drug_pairs[key]['serious_reports'] += is_serious
                drug_pairs[key]['death_reports'] += is_death
                drug_pairs[key]['hospitalization_reports'] += is_hosp

                for rx in reactions:
                    term = rx.get('reactionmeddrapt', '').lower()
                    if term:
                        drug_pairs[key]['reactions'][term] += 1

    return drug_pairs


# Process all downloaded files
all_pairs = defaultdict(lambda: {
    'total_reports': 0,
    'serious_reports': 0,
    'death_reports': 0,
    'hospitalization_reports': 0,
    'reactions': defaultdict(int)
})

json_files = [f'{DATA_DIR}/{f}' for f in os.listdir(DATA_DIR) if f.endswith('.json')]

for filepath in json_files:
    file_pairs = process_fda_file(filepath)
    # Merge into all_pairs
    for key, data in file_pairs.items():
        all_pairs[key]['total_reports'] += data['total_reports']
        all_pairs[key]['serious_reports'] += data['serious_reports']
        all_pairs[key]['death_reports'] += data['death_reports']
        all_pairs[key]['hospitalization_reports'] += data['hospitalization_reports']
        for reaction, count in data['reactions'].items():
            all_pairs[key]['reactions'][reaction] += count

print(f'\n✅ Total unique drug pairs: {len(all_pairs):,}')

In [ ]:
# Convert to DataFrame and create ML features
records = []

for key, data in all_pairs.items():
    drug1, drug2 = key.split('|||')
    total = data['total_reports']
    serious = data['serious_reports']
    deaths = data['death_reports']
    hosp = data['hospitalization_reports']

    # Severity score (0-1)
    severity_score = serious / total if total > 0 else 0
    death_score = deaths / total if total > 0 else 0
    hosp_score = hosp / total if total > 0 else 0

    # Interaction label (binary classification)
    # HIGH: many reports + serious
    # MODERATE: some reports or serious
    # LOW/NONE: few reports
    if total >= 20 and severity_score >= 0.5:
        severity = 'HIGH'
        label = 2
    elif total >= 10 or (total >= 5 and severity_score >= 0.3):
        severity = 'MODERATE'
        label = 1
    else:
        severity = 'LOW'
        label = 0

    # Top reactions
    top_reactions = sorted(data['reactions'].items(), key=lambda x: x[1], reverse=True)[:5]
    top_reaction_str = '|'.join([r[0] for r in top_reactions])

    records.append({
        'drug1': drug1,
        'drug2': drug2,
        'drug_pair': f'{drug1} + {drug2}',
        'total_reports': total,
        'serious_reports': serious,
        'death_reports': deaths,
        'hosp_reports': hosp,
        'severity_score': round(severity_score, 4),
        'death_score': round(death_score, 4),
        'hosp_score': round(hosp_score, 4),
        'top_reactions': top_reaction_str,
        'severity': severity,
        'label': label,
        'source': 'FDA FAERS'
    })

df = pd.DataFrame(records)
df = df.sort_values('total_reports', ascending=False).reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
print(f'\nSeverity distribution:')
print(df['severity'].value_counts())
print(f'\nTop 10 interactions:')
print(df[['drug1','drug2','total_reports','severity']].head(10))

# Save to Drive
df.to_csv(f'{PROJECT_DIR}/data/fda_interactions.csv', index=False)
print(f'\n✅ Saved to {PROJECT_DIR}/data/fda_interactions.csv')

## 🤖 STEP 3: Train XGBoost ML Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from xgboost import XGBClassifier
import joblib
import seaborn as sns
import matplotlib.pyplot as plt

# ── Feature Engineering ─────────────────────────────────────
print('Building features...')

# Encode drug names
le_drug = LabelEncoder()
all_drugs = pd.concat([df['drug1'], df['drug2']]).unique()
le_drug.fit(all_drugs)

df['drug1_encoded'] = le_drug.transform(df['drug1'])
df['drug2_encoded'] = le_drug.transform(df['drug2'])

# Features
FEATURES = [
    'drug1_encoded',
    'drug2_encoded',
    'total_reports',
    'serious_reports',
    'death_reports',
    'hosp_reports',
    'severity_score',
    'death_score',
    'hosp_score'
]

X = df[FEATURES]
y = df['label']  # 0=LOW, 1=MODERATE, 2=HIGH

print(f'Features: {X.shape}')
print(f'Labels: {y.value_counts().to_dict()}')

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# ── Train XGBoost Model ──────────────────────────────────────
print('Training XGBoost model...')

model = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    device='cuda'  # Use GPU!
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

# ── Evaluate ─────────────────────────────────────────────────
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f'\n✅ Model Accuracy: {accuracy:.2%}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred,
    target_names=['LOW', 'MODERATE', 'HIGH']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d',
    xticklabels=['LOW','MODERATE','HIGH'],
    yticklabels=['LOW','MODERATE','HIGH'],
    cmap='Blues')
plt.title('Confusion Matrix - Drug Interaction Classifier')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/models/confusion_matrix.png', dpi=150)
plt.show()

# Feature importance
fi = pd.DataFrame({
    'feature': FEATURES,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=fi, x='importance', y='feature', palette='viridis')
plt.title('Feature Importance - XGBoost Drug Interaction Model')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/models/feature_importance.png', dpi=150)
plt.show()

print(fi)

In [ ]:
# Save model and encoder
joblib.dump(model, f'{PROJECT_DIR}/models/xgboost_interaction_model.pkl')
joblib.dump(le_drug, f'{PROJECT_DIR}/models/drug_label_encoder.pkl')
joblib.dump(FEATURES, f'{PROJECT_DIR}/models/feature_names.pkl')

print('✅ Model saved to Google Drive!')
print(f'  {PROJECT_DIR}/models/xgboost_interaction_model.pkl')
print(f'  {PROJECT_DIR}/models/drug_label_encoder.pkl')

## 📚 STEP 4: Build RAG Vector Database

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import pickle
import numpy as np

print('Loading sentence transformer model...')
# This model converts text to vectors
# FREE, runs on GPU!
embedder = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')
print('✅ Model loaded!')

In [ ]:
# Build RAG documents from FDA data
print('Building RAG documents...')

# Only use pairs with enough evidence (10+ reports)
rag_df = df[df['total_reports'] >= 10].copy()
print(f'Using {len(rag_df):,} drug pairs for RAG')

# Create rich text documents for each interaction
rag_documents = []
rag_metadata = []

for _, row in rag_df.iterrows():
    # Create descriptive document
    doc = (
        f"Drug Interaction: {row['drug1'].title()} and {row['drug2'].title()}. "
        f"Severity: {row['severity']}. "
        f"Evidence: {row['total_reports']} adverse event reports in FDA FAERS database. "
        f"Of these, {row['serious_reports']} were serious, "
        f"{row['death_reports']} resulted in death, "
        f"{row['hosp_reports']} required hospitalization. "
        f"Common adverse reactions: {row['top_reactions'].replace('|', ', ')}. "
        f"Severity score: {row['severity_score']:.2f}. "
        f"Source: FDA Adverse Event Reporting System (FAERS)."
    )

    rag_documents.append(doc)
    rag_metadata.append({
        'drug1': row['drug1'],
        'drug2': row['drug2'],
        'severity': row['severity'],
        'total_reports': int(row['total_reports']),
        'serious_reports': int(row['serious_reports']),
        'death_reports': int(row['death_reports']),
        'top_reactions': row['top_reactions'],
        'source': 'FDA FAERS'
    })

print(f'✅ Created {len(rag_documents):,} RAG documents')
print(f'\nExample document:')
print(rag_documents[0])

In [ ]:
# Create embeddings (GPU accelerated!)
print('Creating embeddings (this may take a few minutes)...')

# Batch encode for speed
embeddings = embedder.encode(
    rag_documents,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f'✅ Embeddings shape: {embeddings.shape}')
print(f'   {embeddings.shape[0]:,} documents')
print(f'   {embeddings.shape[1]} dimensions per embedding')

In [ ]:
# Build FAISS index
print('Building FAISS vector index...')

dimension = embeddings.shape[1]  # 384 for MiniLM

# Normalize for cosine similarity
faiss.normalize_L2(embeddings)

# Create index (Inner Product = cosine similarity after normalization)
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print(f'✅ FAISS index built!')
print(f'   Total vectors: {index.ntotal:,}')

# Test query
print('\nTesting RAG search: warfarin + aspirin')
query = 'warfarin aspirin drug interaction adverse events'
query_embedding = embedder.encode([query], convert_to_numpy=True)
faiss.normalize_L2(query_embedding)

scores, indices = index.search(query_embedding, k=3)

print('Top 3 results:')
for i, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
    meta = rag_metadata[idx]
    print(f'{i}. {meta["drug1"]} + {meta["drug2"]} '
          f'(severity: {meta["severity"]}, score: {score:.3f})')

In [ ]:
# Save FAISS index and metadata
faiss.write_index(index, f'{PROJECT_DIR}/rag/faiss_index.bin')

with open(f'{PROJECT_DIR}/rag/rag_metadata.pkl', 'wb') as f:
    pickle.dump(rag_metadata, f)

with open(f'{PROJECT_DIR}/rag/rag_documents.pkl', 'wb') as f:
    pickle.dump(rag_documents, f)

print('✅ RAG database saved to Google Drive!')
print(f'  {PROJECT_DIR}/rag/faiss_index.bin')
print(f'  {PROJECT_DIR}/rag/rag_metadata.pkl')

## 🧠 STEP 5: LLM Integration with RAG Validation

In [ ]:
# ⚠️ Add your Groq API key here
# Get FREE key: https://console.groq.com
import os
os.environ['GROQ_API_KEY'] = 'your_groq_api_key_here'  # ← CHANGE THIS!

In [ ]:
from groq import Groq
import json

class MedCheckAIPipeline:
    """
    Complete ML + RAG + LLM Pipeline
    ─────────────────────────────────
    1. ML Model predicts interaction probability
    2. RAG retrieves supporting FDA evidence
    3. LLM generates patient-friendly explanation
    4. RAG validates LLM output (no hallucinations!)
    """

    def __init__(self):
        # Load ML model
        self.model = joblib.load(f'{PROJECT_DIR}/models/xgboost_interaction_model.pkl')
        self.le_drug = joblib.load(f'{PROJECT_DIR}/models/drug_label_encoder.pkl')
        self.features = joblib.load(f'{PROJECT_DIR}/models/feature_names.pkl')

        # Load RAG
        self.faiss_index = faiss.read_index(f'{PROJECT_DIR}/rag/faiss_index.bin')
        with open(f'{PROJECT_DIR}/rag/rag_metadata.pkl', 'rb') as f:
            self.rag_metadata = pickle.load(f)
        with open(f'{PROJECT_DIR}/rag/rag_documents.pkl', 'rb') as f:
            self.rag_documents = pickle.load(f)

        # Load embedder
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')

        # LLM client
        self.llm = Groq(api_key=os.environ['GROQ_API_KEY'])

        # Load data for FDA lookup
        self.fda_df = pd.read_csv(f'{PROJECT_DIR}/data/fda_interactions.csv')

        print('✅ MedCheck AI Pipeline loaded!')

    def get_fda_data(self, drug1, drug2):
        """Get FDA data for a drug pair"""
        d1, d2 = sorted([drug1.lower(), drug2.lower()])
        result = self.fda_df[
            (self.fda_df['drug1'] == d1) &
            (self.fda_df['drug2'] == d2)
        ]
        if not result.empty:
            return result.iloc[0].to_dict()
        return None

    def ml_predict(self, drug1, drug2, fda_data=None):
        """ML model prediction"""
        try:
            # Encode drug names
            known_drugs = list(self.le_drug.classes_)

            if drug1.lower() not in known_drugs or drug2.lower() not in known_drugs:
                return {'known': False, 'probability': 0.0, 'label': 0}

            d1_enc = self.le_drug.transform([drug1.lower()])[0]
            d2_enc = self.le_drug.transform([drug2.lower()])[0]

            # Use FDA data if available
            total = fda_data['total_reports'] if fda_data else 0
            serious = fda_data['serious_reports'] if fda_data else 0
            deaths = fda_data['death_reports'] if fda_data else 0
            hosp = fda_data['hosp_reports'] if fda_data else 0
            sev_score = fda_data['severity_score'] if fda_data else 0
            death_score = fda_data['death_score'] if fda_data else 0
            hosp_score = fda_data['hosp_score'] if fda_data else 0

            X = pd.DataFrame([[
                d1_enc, d2_enc, total, serious, deaths,
                hosp, sev_score, death_score, hosp_score
            ]], columns=self.features)

            proba = self.model.predict_proba(X)[0]
            label = self.model.predict(X)[0]

            return {
                'known': True,
                'label': int(label),
                'severity': ['LOW', 'MODERATE', 'HIGH'][label],
                'probabilities': {
                    'LOW': float(proba[0]),
                    'MODERATE': float(proba[1]),
                    'HIGH': float(proba[2])
                },
                'confidence': float(max(proba))
            }
        except Exception as e:
            return {'known': False, 'error': str(e), 'label': 0}

    def rag_retrieve(self, drug1, drug2, top_k=3):
        """Retrieve relevant evidence from RAG database"""
        query = f'{drug1} {drug2} drug interaction adverse effects'
        query_emb = self.embedder.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(query_emb)

        scores, indices = self.faiss_index.search(query_emb, k=top_k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if score > 0.5:  # Relevance threshold
                results.append({
                    'document': self.rag_documents[idx],
                    'metadata': self.rag_metadata[idx],
                    'relevance_score': float(score)
                })

        return results

    def llm_explain(self, drug1, drug2, ml_result, rag_evidence, fda_data):
        """Generate LLM explanation grounded in RAG evidence"""

        # Build context from RAG
        rag_context = '\n'.join([
            f'- {r["document"]}'
            for r in rag_evidence[:3]
        ]) if rag_evidence else 'No direct evidence found in database.'

        fda_info = ''
        if fda_data:
            fda_info = (
                f'FDA FAERS Reports: {fda_data["total_reports"]} total, '
                f'{fda_data["serious_reports"]} serious, '
                f'{fda_data["death_reports"]} deaths. '
                f'Common reactions: {fda_data.get("top_reactions", "N/A")}'
            )

        prompt = f"""You are a clinical pharmacist explaining drug interactions to a patient.
IMPORTANT: Only use the evidence provided below. Do NOT add information not in the evidence.

Drug 1: {drug1.title()}
Drug 2: {drug2.title()}
ML Model Severity Prediction: {ml_result.get('severity', 'UNKNOWN')} 
(confidence: {ml_result.get('confidence', 0):.1%})

FDA FAERS Evidence:
{fda_info if fda_info else 'No FDA data available'}

RAG Database Evidence:
{rag_context}

Based ONLY on the evidence above, provide:
1. A 2-sentence summary of the interaction risk
2. Why this interaction occurs (mechanism, if in evidence)
3. What the patient should do
4. One specific warning sign to watch for

Keep it simple, clear, and under 150 words. 
End with: 'Source: FDA FAERS Database'"""

        try:
            response = self.llm.chat.completions.create(
                model='llama-3.1-8b-instant',
                messages=[{'role': 'user', 'content': prompt}],
                max_tokens=300,
                temperature=0.1  # Low temperature = less hallucination!
            )
            return response.choices[0].message.content
        except Exception as e:
            return f'Could not generate explanation: {e}'

    def rag_validate(self, explanation, drug1, drug2):
        """Validate LLM explanation against RAG database"""
        # Check if explanation is grounded in our data
        evidence = self.rag_retrieve(drug1, drug2, top_k=5)

        if not evidence:
            return {
                'validated': False,
                'reason': 'No evidence found in database',
                'confidence': 0.0
            }

        # Check relevance score of top result
        top_score = evidence[0]['relevance_score'] if evidence else 0

        return {
            'validated': top_score > 0.6,
            'reason': 'Grounded in FDA FAERS data' if top_score > 0.6 else 'Low confidence',
            'confidence': top_score,
            'supporting_evidence_count': len(evidence)
        }

    def check_interaction(self, drug1, drug2):
        """Complete pipeline: ML + RAG + LLM"""

        print(f'\nChecking: {drug1.title()} + {drug2.title()}')
        print('─' * 50)

        # Step 1: Get FDA data
        print('1️⃣ Querying FDA database...')
        fda_data = self.get_fda_data(drug1, drug2)

        # Step 2: ML prediction
        print('2️⃣ Running ML model...')
        ml_result = self.ml_predict(drug1, drug2, fda_data)

        # Step 3: RAG retrieval
        print('3️⃣ Retrieving RAG evidence...')
        rag_evidence = self.rag_retrieve(drug1, drug2)

        # Step 4: LLM explanation
        print('4️⃣ Generating LLM explanation...')
        explanation = self.llm_explain(drug1, drug2, ml_result, rag_evidence, fda_data)

        # Step 5: RAG validation
        print('5️⃣ Validating with RAG...')
        validation = self.rag_validate(explanation, drug1, drug2)

        # Final result
        has_interaction = (
            (fda_data is not None and fda_data['total_reports'] >= 5) or
            (ml_result.get('label', 0) > 0) or
            len(rag_evidence) > 0
        )

        result = {
            'drug1': drug1.title(),
            'drug2': drug2.title(),
            'interaction_found': has_interaction,
            'severity': ml_result.get('severity', 'UNKNOWN'),
            'ml_prediction': ml_result,
            'fda_data': fda_data,
            'rag_evidence': rag_evidence,
            'explanation': explanation,
            'validation': validation,
            'sources': ['FDA FAERS', 'XGBoost ML Model', 'FAISS RAG Database', 'Groq LLM']
        }

        # Display
        print(f'\n📊 RESULT:')
        print(f'  Interaction: {"⚠️ FOUND" if has_interaction else "✅ NONE"}')
        print(f'  Severity: {result["severity"]}')
        print(f'  ML Confidence: {ml_result.get("confidence", 0):.1%}')
        print(f'  RAG Evidence Count: {len(rag_evidence)}')
        print(f'  Validated: {validation["validated"]}')
        print(f'\n📝 Explanation:')
        print(f'  {explanation}')

        return result


# Initialize pipeline
pipeline = MedCheckAIPipeline()
print('✅ Pipeline ready!')

## ✅ STEP 6: Test Complete Pipeline

In [ ]:
# Test 1: Known dangerous interaction
result1 = pipeline.check_interaction('warfarin', 'aspirin')

In [ ]:
# Test 2: Generally safe combination
result2 = pipeline.check_interaction('lisinopril', 'metformin')

In [ ]:
# Test 3: Multiple drugs at once
def check_multiple_drugs(drugs):
    print(f'Checking {len(drugs)} drugs: {drugs}')
    print(f'Total pairs: {len(drugs)*(len(drugs)-1)//2}\n')

    results = []
    for i in range(len(drugs)):
        for j in range(i+1, len(drugs)):
            result = pipeline.check_interaction(drugs[i], drugs[j])
            results.append(result)

    interactions = [r for r in results if r['interaction_found']]
    safe = [r for r in results if not r['interaction_found']]

    print(f'\n📊 SUMMARY:')
    print(f'  Total pairs checked: {len(results)}')
    print(f'  ⚠️ Interactions found: {len(interactions)}')
    print(f'  ✅ Safe pairs: {len(safe)}')

    return results

# Test with 4 drugs
test_drugs = ['warfarin', 'aspirin', 'metformin', 'lisinopril']
all_results = check_multiple_drugs(test_drugs)

## 💾 STEP 7: Export for Deployment

In [ ]:
import shutil

# Create deployment package
DEPLOY_DIR = f'{PROJECT_DIR}/deployment'
os.makedirs(DEPLOY_DIR, exist_ok=True)

# Copy model files
shutil.copy(f'{PROJECT_DIR}/models/xgboost_interaction_model.pkl', DEPLOY_DIR)
shutil.copy(f'{PROJECT_DIR}/models/drug_label_encoder.pkl', DEPLOY_DIR)
shutil.copy(f'{PROJECT_DIR}/models/feature_names.pkl', DEPLOY_DIR)

# Copy RAG files
shutil.copy(f'{PROJECT_DIR}/rag/faiss_index.bin', DEPLOY_DIR)
shutil.copy(f'{PROJECT_DIR}/rag/rag_metadata.pkl', DEPLOY_DIR)

# Save model info
model_info = {
    'model': 'XGBoost Drug Interaction Classifier',
    'accuracy': float(accuracy),
    'training_samples': len(X_train),
    'features': FEATURES,
    'classes': ['LOW', 'MODERATE', 'HIGH'],
    'data_source': 'FDA FAERS',
    'rag_documents': len(rag_documents),
    'embedding_model': 'all-MiniLM-L6-v2'
}

with open(f'{DEPLOY_DIR}/model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print('✅ Deployment package created!')
print(f'\nFiles in {DEPLOY_DIR}:')
for f in os.listdir(DEPLOY_DIR):
    size = os.path.getsize(f'{DEPLOY_DIR}/{f}') / 1e6
    print(f'  {f}: {size:.1f} MB')

In [ ]:
# Final summary
print('=' * 60)
print('🎉 MEDCHECK AI TRAINING COMPLETE!')
print('=' * 60)
print(f'\n📊 Model Performance:')
print(f'  Accuracy: {accuracy:.2%}')
print(f'  Training samples: {len(X_train):,}')
print(f'  Test samples: {len(X_test):,}')
print(f'\n📚 RAG Database:')
print(f'  Documents: {len(rag_documents):,}')
print(f'  Vector dimensions: {dimension}')
print(f'\n🏛️ Data Sources:')
print(f'  FDA FAERS adverse events processed')
print(f'  Total drug pairs: {len(df):,}')
print(f'\n💾 Files saved to Google Drive:')
print(f'  {DEPLOY_DIR}')
print(f'\n🚀 Next Steps:')
print(f'  1. Download files from Google Drive')
print(f'  2. Copy to medcheck/models/ folder')
print(f'  3. Update web/app.py to use pipeline')
print(f'  4. Deploy to Streamlit Cloud!')
print('=' * 60)